<a href="https://colab.research.google.com/github/Lucasgsfaa/previs-o-cancelamento-/blob/main/rede_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import requests
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [9]:



url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

def extração_data(url: str) -> pd.DataFrame:
    # O Pandas consegue abrir e ler URLs diretamente de forma nativa e segura
    return pd.read_csv(url)

df = extração_data(url)

def salvar_data(df: pd.DataFrame, path_saida: str = "df_data.json") -> None:
    # Salvando diretamente no diretório atual sem precisar criar pastas complexas
    df.to_json(path_saida, orient="records", force_ascii=False, indent=4)

In [46]:
#limpando e organixando os dados para a rede
def organizar_data(df:pd.DataFrame)->pd.DataFrame:
 df= df.drop(columns=["customerID"])
 df["TotalCharges"]=pd.to_numeric(df["TotalCharges"], errors ="coerce")
 df["Churn"]=(df["Churn"]=="Yes").astype(int)
 return df


In [35]:
# separando x  y

def separar_xy(df:pd.DataFrame)->pd.DataFrame:
  X =df.drop(columns=["Churn"])
  y= df["Churn"]
  X= pd.get_dummies(X,drop_first=True)
  return X,y



In [67]:
#separando x e y pra treino e teste
def preparar_xy(X,y):
 X_train,X_test,y_train,y_test=train_test_split(
  X,y,test_size=0.2,random_state=SEMENTE,stratify=y
 )


 mediana = X_train["TotalCharges"].median()
 X_train=X_train.fillna({"TotalCharges":mediana})
 X_test=X_test.fillna({"TotalCharges":mediana})


 scaler = StandardScaler()
 X_train=scaler.fit_transform(X_train.astype("float32"))
 X_test=scaler.transform(X_test.astype("float32"))



 return X_train,X_test,y_train.to_numpy(),y_test.to_numpy()


In [75]:
#balancear
SEMENTE=42

def balancear (X,y,semente:int=SEMENTE):
 rng=np.random.default_rng(semente)
 idx_pos= np.where(y==1)[0]
 idx_neg= np.where(y==0)[0]

 faltam = len(idx_neg)-len(idx_pos)
 extras= rng.choice(idx_pos,size=faltam,replace=True)

 idx=np.concatenate([np.arange(len(y)),extras])
 rng.shuffle(idx)
 return X[idx], y[idx]



In [99]:

# treino rede neural
def treinar_rede(X_train,y_train)-> MLPClassifier:
  rede= MLPClassifier(
   hidden_layer_sizes=(32, 16),
   activation="relu",
   alpha=0.01,
   learning_rate_init=0.001,
   max_iter=300,
   early_stopping=True,
   random_state=SEMENTE,
  )
  rede.fit(X_train,y_train)
  print(f'Treino terminou em {rede.n_iter_} épocas')
  return rede


In [106]:
# Avaliar rede neural
def avaliar(rede:MLPClassifier,X_test,y_test)->None:
  prob=rede.predict_proba(X_test)[:,1]
  previsões=(prob>=0.25).astype(int)
  print("\nAcurácia no teste:",round(rede.score(X_test,y_test),3))
  print("\nMatriz de confusão(linhas=real,colunas=previsto):")
  print(confusion_matrix(y_test,previsões))
  print("\nRelatório(classe 1=cancelou):")
  print(classification_report(y_test,previsões,target_names=["ficou","cancelou"]))





In [107]:
#rodando as funções
def main()->None:
 df = extração_data(url)
 df = organizar_data(df)
 X, y = separar_xy(df)
 X_train, X_test, y_train, y_test = preparar_xy(X, y)
 X_train, y_train = balancear(X_train, y_train)
 rede=treinar_rede(X_train, y_train)
 avaliar(rede, X_test, y_test)


if __name__ == "__main__":
    main()




Treino terminou em 33 épocas

Acurácia no teste: 0.747

Matriz de confusão(linhas=real,colunas=previsto):
[[556 479]
 [ 26 348]]

Relatório(classe 1=cancelou):
              precision    recall  f1-score   support

       ficou       0.96      0.54      0.69      1035
    cancelou       0.42      0.93      0.58       374

    accuracy                           0.64      1409
   macro avg       0.69      0.73      0.63      1409
weighted avg       0.81      0.64      0.66      1409

